In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../data/clean/books_clean.csv')
nrow, ncol = df.shape
print(f'There are {nrow} rows and {ncol} columns.')
display(df)

There are 175 rows and 10 columns.


,book_id,upc,title,category,description,lang,image_path,price,rating,product_url
0,998,6957f44c3847a760,Soumission,Fiction,"Dans une France assez proche de la nôtre, un h...",fr,data/raw/images/6957f44c3847a760.jpg,50.10,1,https://books.toscrape.com/catalogue/soumissio...
1,958,b12b89017878a60d,Private Paris (Private #10),Fiction,Paris is burning--and only Private's Jack Morg...,en,data/raw/images/b12b89017878a60d.jpg,47.61,5,https://books.toscrape.com/catalogue/private-p...
2,954,8d455c7539795d2a,"We Love You, Charlie Freeman",Fiction,"The Freeman family--Charles, Laurel, and their...",en,data/raw/images/8d455c7539795d2a.jpg,50.27,5,https://books.toscrape.com/catalogue/we-love-y...
3,946,709822d0b5bcb7f4,Thirst,Fiction,"On a searing summer Friday, Eddie Chapman has ...",en,data/raw/images/709822d0b5bcb7f4.jpg,17.27,5,https://books.toscrape.com/catalogue/thirst_94...
4,939,d01ac97e2b8947c2,The Murder That Never Was (Forensic Instincts #5),Fiction,"Given the opportunity, would you assume someon...",en,data/raw/images/d01ac97e2b8947c2.jpg,54.11,3,https://books.toscrape.com/catalogue/the-murde...
...,...,...,...,...,...,...,...,...,...,...
170,102,b6d3f4f4ee1f6069,H is for Hawk,Nonfiction,When Helen Macdonald's father died suddenly on...,en,data/raw/images/b6d3f4f4ee1f6069.jpg,57.42,5,https://books.toscrape.com/catalogue/h-is-for-...
171,90,0268f149d014b389,Travels with Charley: In Search of America,Nonfiction,An intimate journey across and in search of Am...,en,data/raw/images/0268f149d014b389.jpg,57.82,5,https://books.toscrape.com/catalogue/travels-w...
172,85,6514add13c82b115,The Tumor,Nonfiction,John Grisham says THE TUMOR is the most import...,en,data/raw/images/6514add13c82b115.jpg,41.56,5,https://books.toscrape.com/catalogue/the-tumor...
173,64,09659d1639a1f978,The End of the Jesus Era (An Investigation #1),Nonfiction,God exists!? This may seem unprovable. The tru...,en,data/raw/images/09659d1639a1f978.jpg,14.40,1,https://books.toscrape.com/catalogue/the-end-o...


In [3]:
df["text"] = (df["title"].fillna("") + " " + df["description"].fillna(""))
print(df["text"].isna().sum())
display(df[["book_id", "title", "category", "text"]].head())

0


,book_id,title,category,text
0,998,Soumission,Fiction,Soumission Dans une France assez proche de la ...
1,958,Private Paris (Private #10),Fiction,Private Paris (Private #10) Paris is burning--...
2,954,"We Love You, Charlie Freeman",Fiction,"We Love You, Charlie Freeman The Freeman famil..."
3,946,Thirst,Fiction,"Thirst On a searing summer Friday, Eddie Chapm..."
4,939,The Murder That Never Was (Forensic Instincts #5),Fiction,The Murder That Never Was (Forensic Instincts ...


In [4]:
#Merging the split with the df on book_id
#Split.csv is created in notebooks/d_modeling.ipynb (80/20 stratified split, random_state=42).
split_df = pd.read_csv('../data/features/split.csv')
df = df.merge(split_df, on = "book_id")
train_texts = df[df["split"] == "train"]["text"] 


In [5]:
#Build the first feature set file in which each row represents the book with occurences of each word
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words="english", min_df=2)

# Learn the vocabulary and count words in each book's text
vectorizer.fit(train_texts) #learn word columns from training books only
word_counts = vectorizer.transform(df["text"]) #count those words in all books
features = pd.DataFrame(
    word_counts.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=df["book_id"]
) #create a df in which each row is like a vector, the first entry is the book_id, the rest is 
 #the number of occurences of the word of that column 

display(features)
features.to_csv(
     "../data/features/features_word_counts.csv",
    index=True,
    index_label="book_id"
) #saves the df into a csv file for training


,000,10,100,11,13,1942,1945,1950s,1955,1970s,...,wrote,wry,year,years,yes,york,young,younger,youngest,youth
book_id,,,,,,,,,,,,,,,,,,,,,
998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
958,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
954,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,1,0,0,0
946,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
939,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
90,0,0,0,0,0,0,0,0,0,0,...,1,0,0,1,0,1,0,0,0,0
85,0,0,0,0,0,0,0,0,0,0,...,0,0,2,0,0,0,0,0,0,0


In [6]:
#Build the second feature set file in which each vector/row represents
#the book and the importance of each word 

from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(stop_words="english", min_df=2)
tfidf_vectorizer.fit(train_texts) #learns vocab and calculate rarity weight of each word from training books
tfidf_matrix = tfidf_vectorizer.transform(df["text"]) #calculate the tf idf value for each word in every book
features_tfidf = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns = tfidf_vectorizer.get_feature_names_out(),
    index = df["book_id"],
)
display(features_tfidf)
features_tfidf.to_csv(
     "../data/features/features_tfidf.csv",
    index=True,
    index_label="book_id"
)


train_df = df[df["split"] == "train"]

fiction_ids = train_df[train_df["category"] == "Fiction"]["book_id"]
nonfiction_ids = train_df[train_df["category"] == "Nonfiction"]["book_id"]

top_fiction = features_tfidf.loc[fiction_ids].mean().nlargest(15)
top_nonfiction = features_tfidf.loc[nonfiction_ids].mean().nlargest(15)

#15 Highest-weighted words for Fiction vs Non-fiction:
print("Top 15 Fiction words (mean TF-IDF):")
print(top_fiction)
print("\nTop 15 Nonfiction words (mean TF-IDF):")
print(top_nonfiction)

,000,10,100,11,13,1942,1945,1950s,1955,1970s,...,wrote,wry,year,years,yes,york,young,younger,youngest,youth
book_id,,,,,,,,,,,,,,,,,,,,,
998,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.000000,0.0000,0.0,0.000000,0.000000,0.0,0.0,0.0
958,0.0,0.090247,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.000000,0.0000,0.0,0.000000,0.054418,0.0,0.0,0.0
954,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.086130,0.0000,0.0,0.000000,0.077598,0.0,0.0,0.0
946,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.000000,0.0000,0.0,0.000000,0.000000,0.0,0.0,0.0
939,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.000000,0.0000,0.0,0.000000,0.000000,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.000000,0.0000,0.0,0.000000,0.000000,0.0,0.0,0.0
90,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.099428,0.0,0.000000,0.0498,0.0,0.057814,0.000000,0.0,0.0,0.0
85,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.131507,0.0000,0.0,0.000000,0.000000,0.0,0.0,0.0


Top 15 Fiction words (mean TF-IDF):
love        0.044734
novel       0.043114
family      0.040106
life        0.033826
new         0.033738
secrets     0.027745
daughter    0.027263
world       0.026863
man         0.026681
young       0.026206
old         0.025080
story       0.024489
home        0.022843
wife        0.022649
son         0.022606
dtype: float64

Top 15 Nonfiction words (mean TF-IDF):
life       0.041157
book       0.032649
world      0.032248
new        0.030638
god        0.023447
love       0.023193
way        0.022614
people     0.022511
time       0.021570
story      0.021510
lives      0.021415
america    0.020891
like       0.020577
readers    0.018817
work       0.018632
dtype: float64


In [7]:
#Build the third feature set, in which each row
#represents the cover image's RGB color, standard deviation, and distribution of color intensity
import numpy as np
from PIL import Image
image_rows = []
for path in df["image_path"]:
    img = Image.open("../" + path).convert("RGB").resize((64, 64)) 
    #rescaling the image to 64x64, img here is a Pillow Image object
    arr = np.array(img) / 255.0 
    #converts the img into the 3-dimensional array of numbers (64 rows, 64 columns, 3 color values)
    #arr[row, column, red/blue/green] = how red/blue/green this pixel at (row, column) is
    #divide each entry at (row, column, color) by 255 -> value now ranges from 0 to 1 in decimal

    red, green, blue = arr[:, :, 0], arr[:, :, 1], arr[:, :, 2]
    #each variable is referenced to a two dimensional array in which each value is how 
    #red/blue/green the pixel is
    
    r_mean, g_mean, b_mean = red.mean(), green.mean(), blue.mean() 
    #how red/blue/green the picture in 64x64 is in average
    
    r_std, g_std, b_std = red.std(), green.std(), blue.std()
    #how different is the red/blue/green of every pixel from the mean red/blue/green of the image.
    
    r_hist = np.histogram(red, bins=8, range=(0.0, 1.0))[0] / red.size
    g_hist = np.histogram(green, bins=8, range=(0.0, 1.0))[0] / green.size
    b_hist = np.histogram(blue, bins=8, range=(0.0, 1.0))[0] / blue.size
    #each histogram includes 8 bins, each showing what fraction of pixels in the cover is in
    #that range of color intensity
    
    
    row = [r_mean, g_mean, b_mean, r_std, g_std, b_std] + list(r_hist) + list(g_hist) + list(b_hist)
    image_rows.append(row)



features_images = pd.DataFrame(
    image_rows, 
    columns = ["r_mean", "g_mean", "b_mean", "r_std", "g_std", "b_std"]
        + [f"r_bin_{i}" for i in range(8)]
        + [f"g_bin_{i}" for i in range(8)]
        + [f"b_bin_{i}" for i in range(8)],
    index=df["book_id"]
)

display(features_images)

features_images.to_csv(
    "../data/features/features_images.csv", 
    index=True,
    index_label="book_id"
)

,r_mean,g_mean,b_mean,r_std,g_std,b_std,r_bin_0,r_bin_1,r_bin_2,r_bin_3,...,g_bin_6,g_bin_7,b_bin_0,b_bin_1,b_bin_2,b_bin_3,b_bin_4,b_bin_5,b_bin_6,b_bin_7
book_id,,,,,,,,,,,,,,,,,,,,,
998,0.929957,0.902109,0.658482,0.019284,0.051160,0.034236,0.000000,0.000000,0.000000,0.000000,...,0.032715,0.930908,0.000000,0.000000,0.000000,0.011963,0.055908,0.932129,0.000000,0.000000
958,0.568638,0.435224,0.446772,0.334272,0.349310,0.345395,0.133301,0.110596,0.064453,0.087891,...,0.059570,0.186523,0.192627,0.229004,0.076416,0.052246,0.131348,0.065186,0.060059,0.193115
954,0.167210,0.432359,0.288279,0.231820,0.169555,0.147504,0.693848,0.102539,0.063232,0.022705,...,0.053223,0.039795,0.000000,0.581055,0.267578,0.052002,0.038818,0.033691,0.018311,0.008545
946,0.046018,0.038812,0.022676,0.128279,0.104136,0.044004,0.899658,0.053467,0.016357,0.006836,...,0.000732,0.000000,0.947754,0.051514,0.000732,0.000000,0.000000,0.000000,0.000000,0.000000
939,0.675732,0.184586,0.202058,0.252238,0.197278,0.193549,0.072998,0.055420,0.036133,0.050781,...,0.011475,0.027344,0.239014,0.537842,0.103027,0.045410,0.022949,0.012695,0.012451,0.026611
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102,0.562588,0.531867,0.455397,0.363473,0.343955,0.297669,0.214844,0.049561,0.069580,0.072021,...,0.116699,0.273926,0.223145,0.073242,0.095459,0.123291,0.078857,0.130371,0.273438,0.002197
90,0.776834,0.759632,0.604111,0.254363,0.273295,0.213679,0.000000,0.017822,0.079346,0.122070,...,0.077637,0.534180,0.000000,0.059326,0.180420,0.103271,0.076172,0.148193,0.432617,0.000000
85,0.461445,0.563307,0.503593,0.279821,0.178302,0.227992,0.190674,0.061523,0.106201,0.134766,...,0.151611,0.022949,0.013672,0.159668,0.196045,0.102539,0.212891,0.152588,0.109131,0.053467
